# 01D — Lead Feature Engineering Optimizer
## bd_replica_crm · optimizar `refresh_historical_features()`

Confirmado:
- 205,947 evidencias pendientes de features.
- 1,930 pendientes dentro de la ventana live.
- `features_refreshed_at` está NULL en todo el universo.

Objetivo: comparar el método correlacionado actual contra una alternativa optimizada y diseñar la separación BACKFILL vs LIVE.
Por defecto es read-only.


In [10]:
from __future__ import annotations
import sys, time
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
from replica_cygnus.lead_scoring.config import load_lead_scoring_config

settings=load_settings(PROJECT_ROOT)
config_path=PROJECT_ROOT/"config"/"lead_scoring.yml"
if not config_path.exists():
    config_path=PROJECT_ROOT/"config"/"lead_scoring.example.yml"

cfg=load_lead_scoring_config(config_path)
conn=connect_postgres(settings)

pd.set_option("display.max_columns",150)
pd.set_option("display.max_rows",150)
pd.set_option("display.width",220)

def df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)

SEP_H=int(cfg.sep_horizon_days)
MINUTA_H=int(cfg.minuta_horizon_days)
SCORE_WINDOW=int(cfg.score_window_days)

print("DB:",settings.postgres.database)
print("score_window_days:",SCORE_WINDOW)


DB: medallio_dw
score_window_days: 14


## 1. Estado confirmado


In [11]:
status=df(f"""
SELECT
 COUNT(*) AS total,
 COUNT(*) FILTER (WHERE features_refreshed_at IS NULL) AS pending,
 COUNT(*) FILTER (
   WHERE features_refreshed_at IS NULL
     AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ) AS pending_live,
 MIN(decision_at) FILTER (WHERE features_refreshed_at IS NULL) AS min_pending,
 MAX(decision_at) FILTER (WHERE features_refreshed_at IS NULL) AS max_pending
FROM features.lead_evidence
""")
status.T


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,0
total,205947
pending,205947
pending_live,1930
min_pending,2019-10-02 23:08:14.794272+00:00
max_pending,2026-09-07 00:04:08.133367+00:00


## 2. Índices actuales


In [12]:
lead_indexes=df("""
SELECT indexname,indexdef
FROM pg_indexes
WHERE schemaname='features' AND tablename='lead_evidence'
ORDER BY indexname
""")
lead_indexes


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,indexname,indexdef
0,ix_lead_evidence_advisor_time,CREATE INDEX ix_lead_evidence_advisor_time ON ...
1,ix_lead_evidence_decision_at,CREATE INDEX ix_lead_evidence_decision_at ON f...
2,ix_lead_evidence_document_time,CREATE INDEX ix_lead_evidence_document_time ON...
3,ix_lead_evidence_project_time,CREATE INDEX ix_lead_evidence_project_time ON ...
4,lead_evidence_pkey,CREATE UNIQUE INDEX lead_evidence_pkey ON feat...


## 3. Query método actual (read-only)


In [13]:
def current_query(limit_n=100,live_only=True):
    scope="WHERE e.features_refreshed_at IS NULL"
    if live_only:
        scope+=f" AND e.decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')"
    return f"""
WITH target AS (
  SELECT *
  FROM features.lead_evidence e
  {scope}
  ORDER BY decision_at,evidence_key
  LIMIT {int(limit_n)}
)
SELECT
 e.evidence_key,e.decision_at,e.documento_cliente,e.codigo_proyecto,e.asesor,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
           WHERE p.documento_cliente=e.documento_cliente
             AND p.decision_at<e.decision_at
             AND p.decision_at>=e.decision_at-interval '90 days'),0)
   AS client_prior_assignments_90d,
 (SELECT EXTRACT(EPOCH FROM (e.decision_at-MAX(p.decision_at)))/86400.0
    FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at)
   AS days_since_previous_assignment,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
           WHERE p.codigo_proyecto=e.codigo_proyecto
             AND p.decision_at<e.decision_at
             AND p.decision_at>=e.decision_at-interval '90 days'),0)
   AS project_leads_90d,
 (SELECT AVG(p.separacion_14d::double precision)
    FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at)
   AS project_sep_rate_90d,
 (SELECT AVG(p.minuta_60d::double precision)
    FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at)
   AS project_minuta_rate_180d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE COALESCE((
   SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) END
   AS advisor_leads_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (
   SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) END
   AS advisor_sep_rate_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (
   SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) END
   AS advisor_minuta_rate_180d,
 (SELECT AVG(p.separacion_14d::double precision)
  FROM features.lead_evidence p
  WHERE p.decision_at<e.decision_at
    AND p.decision_at>=e.decision_at-interval '90 days'
    AND p.separacion_14d IS NOT NULL
    AND p.decision_at+interval '{SEP_H} days'<=e.decision_at)
   AS global_sep_rate_90d,
 (SELECT AVG(p.minuta_60d::double precision)
  FROM features.lead_evidence p
  WHERE p.decision_at<e.decision_at
    AND p.decision_at>=e.decision_at-interval '180 days'
    AND p.minuta_60d IS NOT NULL
    AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at)
   AS global_minuta_rate_180d
FROM target e
ORDER BY e.decision_at,e.evidence_key
"""


## 4. Query optimizada con LATERAL


In [14]:
def optimized_query(limit_n=100,live_only=True):
    scope="WHERE e.features_refreshed_at IS NULL"
    if live_only:
        scope+=f" AND e.decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')"
    return f"""
WITH target AS (
  SELECT *
  FROM features.lead_evidence e
  {scope}
  ORDER BY decision_at,evidence_key
  LIMIT {int(limit_n)}
)
SELECT
 e.evidence_key,e.decision_at,e.documento_cliente,e.codigo_proyecto,e.asesor,
 COALESCE(ch.client_prior_assignments_90d,0) AS client_prior_assignments_90d,
 ch.days_since_previous_assignment,
 COALESCE(ph.project_leads_90d,0) AS project_leads_90d,
 ph.project_sep_rate_90d,
 ph.project_minuta_rate_180d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE COALESCE(ah.advisor_leads_90d,0) END AS advisor_leads_90d,
 ah.advisor_sep_rate_90d,
 ah.advisor_minuta_rate_180d,
 gh.global_sep_rate_90d,
 gh.global_minuta_rate_180d
FROM target e
LEFT JOIN LATERAL (
  SELECT
   COUNT(*) FILTER (WHERE p.decision_at>=e.decision_at-interval '90 days') AS client_prior_assignments_90d,
   EXTRACT(EPOCH FROM (e.decision_at-MAX(p.decision_at)))/86400.0 AS days_since_previous_assignment
  FROM features.lead_evidence p
  WHERE p.documento_cliente=e.documento_cliente
    AND p.decision_at<e.decision_at
) ch ON TRUE
LEFT JOIN LATERAL (
  SELECT
   COUNT(*) FILTER (WHERE p.decision_at>=e.decision_at-interval '90 days') AS project_leads_90d,
   AVG(p.separacion_14d::double precision) FILTER (
      WHERE p.decision_at>=e.decision_at-interval '90 days'
        AND p.separacion_14d IS NOT NULL
        AND p.decision_at+interval '{SEP_H} days'<=e.decision_at
   ) AS project_sep_rate_90d,
   AVG(p.minuta_60d::double precision) FILTER (
      WHERE p.decision_at>=e.decision_at-interval '180 days'
        AND p.minuta_60d IS NOT NULL
        AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at
   ) AS project_minuta_rate_180d
  FROM features.lead_evidence p
  WHERE p.codigo_proyecto=e.codigo_proyecto
    AND p.decision_at<e.decision_at
    AND p.decision_at>=e.decision_at-interval '180 days'
) ph ON TRUE
LEFT JOIN LATERAL (
  SELECT
   COUNT(*) FILTER (WHERE p.decision_at>=e.decision_at-interval '90 days') AS advisor_leads_90d,
   AVG(p.separacion_14d::double precision) FILTER (
      WHERE p.decision_at>=e.decision_at-interval '90 days'
        AND p.separacion_14d IS NOT NULL
        AND p.decision_at+interval '{SEP_H} days'<=e.decision_at
   ) AS advisor_sep_rate_90d,
   AVG(p.minuta_60d::double precision) FILTER (
      WHERE p.decision_at>=e.decision_at-interval '180 days'
        AND p.minuta_60d IS NOT NULL
        AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at
   ) AS advisor_minuta_rate_180d
  FROM features.lead_evidence p
  WHERE e.asesor IS NOT NULL
    AND p.asesor=e.asesor
    AND p.decision_at<e.decision_at
    AND p.decision_at>=e.decision_at-interval '180 days'
) ah ON TRUE
LEFT JOIN LATERAL (
  SELECT
   AVG(p.separacion_14d::double precision) FILTER (
      WHERE p.decision_at>=e.decision_at-interval '90 days'
        AND p.separacion_14d IS NOT NULL
        AND p.decision_at+interval '{SEP_H} days'<=e.decision_at
   ) AS global_sep_rate_90d,
   AVG(p.minuta_60d::double precision) FILTER (
      WHERE p.decision_at>=e.decision_at-interval '180 days'
        AND p.minuta_60d IS NOT NULL
        AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at
   ) AS global_minuta_rate_180d
  FROM features.lead_evidence p
  WHERE p.decision_at<e.decision_at
    AND p.decision_at>=e.decision_at-interval '180 days'
) gh ON TRUE
ORDER BY e.decision_at,e.evidence_key
"""


## 5. Benchmark progresivo


In [15]:
BENCH_SIZES=[100,500,1000]
rows=[]
for n in BENCH_SIZES:
    for name,fn in [("current",current_query),("optimized",optimized_query)]:
        t0=time.perf_counter()
        try:
            out=df(fn(n,True))
            sec=time.perf_counter()-t0
            rows.append({"n":n,"method":name,"seconds":sec,"rows":len(out),"rows_per_second":len(out)/sec if sec else np.nan})
            print(f"{name} n={n}: {sec:.3f}s")
        except Exception as exc:
            conn.rollback()
            rows.append({"n":n,"method":name,"seconds":np.nan,"rows":0,"error":repr(exc)})
benchmark=pd.DataFrame(rows)
benchmark


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


current n=100: 2.599s


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


optimized n=100: 7.136s


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


...

C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


current n=1000: 55.802s


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


optimized n=1000: 68.889s


,n,method,seconds,rows,rows_per_second
0,100,current,2.599077,100,38.475200
1,100,optimized,7.136116,100,14.013225
2,500,current,29.805179,500,16.775608
3,500,optimized,35.735079,500,13.991854
4,1000,current,55.802219,1000,17.920434
5,1000,optimized,68.889087,1000,14.516087


## 6. Speedup


In [16]:
piv=benchmark.pivot(index="n",columns="method",values="seconds").reset_index()
if {"current","optimized"}.issubset(piv.columns):
    piv["speedup_x"]=piv["current"]/piv["optimized"]
piv


method,n,current,optimized,speedup_x
0,100,2.599077,7.136116,0.364214
1,500,29.805179,35.735079,0.834059
2,1000,55.802219,68.889087,0.810030


## 7. Equivalencia funcional


In [17]:
N_CHECK=100
a=df(current_query(N_CHECK,True))
b=df(optimized_query(N_CHECK,True))

feature_cols=[
"client_prior_assignments_90d","days_since_previous_assignment",
"project_leads_90d","project_sep_rate_90d","project_minuta_rate_180d",
"advisor_leads_90d","advisor_sep_rate_90d","advisor_minuta_rate_180d",
"global_sep_rate_90d","global_minuta_rate_180d"
]

m=a.merge(b,on="evidence_key",suffixes=("_current","_optimized"))
checks=[]
for c in feature_cols:
    x=pd.to_numeric(m[f"{c}_current"],errors="coerce")
    y=pd.to_numeric(m[f"{c}_optimized"],errors="coerce")
    ok=np.isclose(x.fillna(-999999),y.fillna(-999999),rtol=1e-9,atol=1e-9)
    checks.append({"feature":c,"match_pct":ok.mean(),"rows":len(ok),"max_abs_diff":float((x-y).abs().max()) if len(x) else np.nan})
equivalence=pd.DataFrame(checks)
equivalence


C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,feature,match_pct,rows,max_abs_diff
0,client_prior_assignments_90d,1.0,100,0.0
1,days_since_previous_assignment,1.0,100,0.0
2,project_leads_90d,1.0,100,0.0
3,project_sep_rate_90d,1.0,100,0.0
4,project_minuta_rate_180d,1.0,100,0.0
5,advisor_leads_90d,1.0,100,NaN
6,advisor_sep_rate_90d,1.0,100,NaN
7,advisor_minuta_rate_180d,1.0,100,NaN
8,global_sep_rate_90d,1.0,100,0.0
9,global_minuta_rate_180d,1.0,100,0.0


## 8. EXPLAIN comparativo


In [18]:
for name,q in [("CURRENT",current_query(100,True)),("OPTIMIZED",optimized_query(100,True))]:
    print("\n###",name)
    p=df("EXPLAIN "+q)
    print("\n".join(p.iloc[:,0].astype(str)))



### CURRENT
Subquery Scan on e  (cost=3.87..753315.15 rows=100 width=194)
  ->  Limit  (cost=3.87..351.81 rows=100 width=380)
        ->  Incremental Sort  (cost=3.87..7334.97 rows=2107 width=380)
              Sort Key: e_1.decision_at, e_1.evidence_key
              Presorted Key: e_1.decision_at
              ->  Index Scan Backward using ix_lead_evidence_decision_at on lead_evidence e_1  (cost=0.42..7240.19 rows=2107 width=380)
                    Index Cond: (decision_at >= (CURRENT_DATE - '14 days'::interval))
                    Filter: (features_refreshed_at IS NULL)
  SubPlan 1
    ->  Aggregate  (cost=8.45..8.46 rows=1 width=8)
          ->  Index Only Scan using ix_lead_evidence_document_time on lead_evidence p  (cost=0.42..8.44 rows=1 width=0)
                Index Cond: ((documento_cliente = e.documento_cliente) AND (decision_at < e.decision_at) AND (decision_at >= (e.decision_at - '90 days'::interval)))
  SubPlan 3
    ->  Result  (cost=8.44..8.46 rows=1 width=32)
      

C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_13240\853950271.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


## 9. Full LIVE benchmark opcional


In [19]:
RUN_FULL_LIVE=False
if RUN_FULL_LIVE:
    n=int(status.iloc[0]["pending_live"])
    full=[]
    for name,fn in [("current",current_query),("optimized",optimized_query)]:
        t0=time.perf_counter()
        out=df(fn(n,True))
        sec=time.perf_counter()-t0
        full.append({"method":name,"rows":len(out),"seconds":sec,"minutes":sec/60,"rows_per_second":len(out)/sec if sec else np.nan})
    full_live=pd.DataFrame(full)
    display(full_live)
else:
    print("RUN_FULL_LIVE=False")


RUN_FULL_LIVE=False


## 10. Estimación de backfill completo


In [20]:
est=[]
pending=int(status.iloc[0]["pending"])
for method in benchmark["method"].unique():
    x=benchmark[(benchmark["method"]==method)&benchmark["rows_per_second"].notna()&benchmark["rows_per_second"].gt(0)]
    if len(x):
        r=x.sort_values("n").iloc[-1]
        rps=float(r["rows_per_second"])
        sec=pending/rps
        est.append({"method":method,"rps_reference":rps,"estimated_hours_full_backfill":sec/3600})
estimates=pd.DataFrame(est)
estimates


,method,rps_reference,estimated_hours_full_backfill
0,current,17.920434,3.192305
1,optimized,14.516087,3.940972


## 11. Índices candidatos


In [21]:
index_candidates=pd.DataFrame([
{"index":"(documento_cliente, decision_at)","supports":"historial cliente"},
{"index":"(codigo_proyecto, decision_at)","supports":"ventanas proyecto 90/180d"},
{"index":"(asesor, decision_at)","supports":"ventanas asesor 90/180d"},
{"index":"(decision_at)","supports":"ventanas globales + live"},
])
index_candidates


,index,supports
0,"(documento_cliente, decision_at)",historial cliente
1,"(codigo_proyecto, decision_at)",ventanas proyecto 90/180d
2,"(asesor, decision_at)",ventanas asesor 90/180d
3,(decision_at),ventanas globales + live


## 12. Arquitectura recomendada


```text
BACKFILL
205,947 filas históricas
→ ejecutar una sola vez
→ features point-in-time completas

LIVE
~1,930 filas recientes
→ WHERE features_refreshed_at IS NULL
→ solo ventana reciente
→ score
→ recommendation
→ action
→ outcome
```

El objetivo es que `41_lead_scoring_live.bat` nunca recalcule todo el histórico.


## 13. Gates


In [22]:
gates=[]
def add(name,passed,detail):
    gates.append({"gate":name,"status":"PASS" if passed else "FAIL","detail":detail})

if len(equivalence):
    mm=float(equivalence["match_pct"].min())
    add("Equivalencia funcional",mm>=0.999,f"min_match={mm:.3%}")

pv=benchmark.pivot(index="n",columns="method",values="seconds").dropna()
if len(pv) and {"current","optimized"}.issubset(pv.columns):
    r=pv.iloc[-1]
    sx=float(r["current"]/r["optimized"])
    add("Performance",sx>=1.5,f"speedup={sx:.2f}x")

add("LIVE incremental",int(status.iloc[0]["pending_live"])>0,f"pending_live={int(status.iloc[0]['pending_live']):,}")

gate_table=pd.DataFrame(gates)
gate_table


,gate,status,detail
0,Equivalencia funcional,PASS,min_match=100.000%
1,Performance,FAIL,speedup=0.81x
2,LIVE incremental,PASS,"pending_live=1,930"


## 14. Smart insights


In [23]:
print("=== FEATURE ENGINEERING OPTIMIZER ===")
print(f"1. Pendientes históricos: {int(status.iloc[0]['pending']):,}")
print(f"2. Pendientes live: {int(status.iloc[0]['pending_live']):,}")

if len(equivalence):
    print(f"3. Equivalencia mínima: {equivalence['match_pct'].min():.1%}")

if len(pv) and {"current","optimized"}.issubset(pv.columns):
    r=pv.iloc[-1]
    print(f"4. Speedup en mayor benchmark: {float(r['current']/r['optimized']):.2f}x")


=== FEATURE ENGINEERING OPTIMIZER ===
1. Pendientes históricos: 205,947
2. Pendientes live: 1,930
3. Equivalencia mínima: 100.0%
4. Speedup en mayor benchmark: 0.81x


## 15. Escritura deshabilitada


In [24]:
APPLY_CHANGES=False
print("APPLY_CHANGES =",APPLY_CHANGES)
print("Este notebook no modifica features.lead_evidence.")


APPLY_CHANGES = False
Este notebook no modifica features.lead_evidence.


In [25]:
conn.close(); print("Conexión cerrada.")


Conexión cerrada.
